# Hotelbooking med Prioritetsmedlems Middleware

Denne notesbog demonstrerer **funktionsbaseret middleware** ved brug af Microsoft Agent Framework. Vi bygger videre på eksemplet med betinget workflow ved at tilføje et middleware-lag, som giver prioriterede medlemmer særlige privilegier.

## Hvad Du Vil Lære:
1. **Funktionsbaseret Middleware**: Opfang og modificer funktionsresultater
2. **Kontekstadgang**: Læs og modificer `context.result` efter udførelse
3. **Implementering af Forretningslogik**: Fordele for prioriterede medlemmer
4. **Resultat Overstyring**: Ændre funktionsudfald baseret på brugerstatus
5. **Samme Workflow, Forskellige Resultater**: Middleware-drevne adfærdsændringer

## Workflow Arkitektur med Middleware:

```
User Input: "I want to book a hotel in Paris"
                    ↓
        [availability_agent]
        - Calls hotel_booking tool
        - 🌟 priority_check middleware intercepts
        - Checks user membership status
        - IF priority + no rooms → Override to available!
        - Returns BookingCheckResult
                    ↓
        Conditional Routing
           /                    \
    [has_availability]    [no_availability]
          ↓                      ↓
    [booking_agent]        [alternative_agent]
    (Priority override!)   (Regular users)
          ↓                      ↓
       [display_result executor]
```

## Nøgleforskel fra Betinget Workflow:

**Uden Middleware** (14-conditional-workflow.ipynb):
- Paris har ingen værelser → Ruter til alternative_agent

**Med Middleware** (denne notesbog):
- Almindelig bruger + Paris → Ingen værelser → Ruter til alternative_agent
- Prioritetsbruger + Paris → 🌟 Middleware overstyrer! → Ledigt → Ruter til booking_agent

## Forudsætninger:
- Microsoft Agent Framework installeret
- Forståelse af betingede workflows (se 14-conditional-workflow.ipynb)
- GitHub token eller OpenAI API nøgle
- Grundlæggende forståelse af middleware mønstre


In [ ]:
import asyncio
import json
import os
from collections.abc import Awaitable, Callable
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    FunctionInvocationContext,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

## Trin 1: Definér Pydantic-modeller for strukturerede output

Disse modeller definerer **skemaet**, som agenter vil returnere. Vi har tilføjet et `priority_override`-felt for at spore, når middleware ændrer tilgængelighedsresultatet.


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str
    # Tracks if middleware overrode the result. The Azure structured-output
    # contract requires every property to be in the JSON schema's `required`
    # array, so we cannot give this a default value the way the original
    # notebook did.
    priority_override: bool


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check with priority_override)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## Trin 2: Definer Prioritetsmedlemmer Database

Til denne demo vil vi simulere en database for prioritetsmedlemskab. I produktion ville dette forespørge en rigtig database eller API.

**Prioritetsmedlemmer:**
- `alice@example.com` - VIP-medlem
- `bob@example.com` - Premium-medlem  
- `priority_user` - Testkonto


In [ ]:
# Simulated priority members database
PRIORITY_MEMBERS = {
    "alice@example.com",
    "bob@example.com",
    "priority_user",
}

# Global variable to track current user (in real app, use proper session management)
current_user_id = "regular_user"  # Default: regular user


def set_user(user_id: str):
    """Set the current user for the session."""
    global current_user_id
    current_user_id = user_id
    is_priority = user_id in PRIORITY_MEMBERS
    status = "🌟 PRIORITY MEMBER" if is_priority else "👤 Regular User"

    display(
        HTML(f"""
        <div style='padding: 15px; background: {"linear-gradient(135deg, #FFD700 0%, #FFA500 100%)" if is_priority else "#e3f2fd"}; 
                    border-left: 4px solid {"#FF6B35" if is_priority else "#2196f3"}; border-radius: 4px; margin: 10px 0;'>
            <strong>👤 Current User Set:</strong> {user_id}<br>
            <strong>Status:</strong> {status}
        </div>
    """)
    )


print("✅ Priority members database created")
print(f"   Priority members: {len(PRIORITY_MEMBERS)} users")

## Trin 3: Opret hotelbookingsværktøjet

Det samme som den betingede arbejdsgang, men nu vil det blive opfanget af middleware!


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## Trin 4: 🌟 Opret Priority Check Middleware (DEN VIGTIGSTE FUNKTION!)

Dette er **kernefunktionaliteten** i denne notesbog. Middleware:

1. **Afgriber** kaldet til funktionen hotel_booking
2. **Udfører** funktionen normalt ved at kalde `next(context)`
3. **Undersøger** resultatet i `context.result`
4. **Overskriver** resultatet, hvis brugeren er prioritet, og der ikke er ledige værelser
5. **Returnerer** det ændrede resultat tilbage til agenten

**Nøglemønster:**
```python
async def my_middleware(context, next):
    await next(context)  # Udfør funktion
    # Nu indeholder context.result funktionens output
    if some_condition:
        context.result = new_value  # Overskriv!
```


In [ ]:
async def priority_check_middleware(
    context: FunctionInvocationContext,
    next: Callable[[FunctionInvocationContext], Awaitable[None]],
) -> None:
    """
    Function middleware that overrides hotel_booking results for priority members.
    
    Workflow:
    1. Let the function execute normally
    2. Check if user is a priority member
    3. If priority + no availability → Override to make rooms available!
    4. Agent will then route to booking path instead of alternative path
    """
    function_name = context.function.name

    display(
        HTML(f"""
        <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Middleware:</strong> Intercepting {function_name}...
        </div>
    """)
    )

    # Execute the original function
    await next(context)

    # Now inspect and potentially modify the result
    if context.result and function_name == "hotel_booking":
        result_data = json.loads(context.result)
        destination = result_data.get("destination", "")
        has_availability = result_data.get("has_availability", False)

        # Check if user is priority member
        is_priority = current_user_id in PRIORITY_MEMBERS

        # Override logic: Priority member + no availability → Make available!
        if is_priority and not has_availability:
            display(
                HTML(f"""
                <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); 
                            border-radius: 8px; margin: 10px 0; box-shadow: 0 4px 12px rgba(255,165,0,0.4);'>
                    <h3 style='margin: 0 0 10px 0; color: #333;'>🌟 PRIORITY OVERRIDE ACTIVATED! 🌟</h3>
                    <p style='margin: 0; color: #555; line-height: 1.6;'>
                        <strong>User:</strong> {current_user_id}<br>
                        <strong>Status:</strong> VIP Priority Member<br>
                        <strong>Action:</strong> Overriding "No Availability" for {destination}<br>
                        <strong>Result:</strong> ✅ Rooms now available for priority booking!
                    </p>
                </div>
            """)
            )

            # Override the result!
            result_data["has_availability"] = True
            result_data["priority_override"] = True
            context.result = json.dumps(result_data)

        elif not has_availability:
            display(
                HTML(f"""
                <div style='padding: 12px; background: #ffebee; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>ℹ️ Middleware:</strong> No priority override (user: {current_user_id})
                </div>
            """)
            )


print("✅ priority_check_middleware created")
print("   - Intercepts hotel_booking function")
print("   - Overrides availability for priority members")

## Trin 5: Definer Betingelsesfunktioner til Routing

De samme betingelsesfunktioner som i den betingede arbejdsgang - de inspicerer den strukturerede output for at bestemme routing.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available (including priority overrides!)."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        # Check if this was a priority override
        override_indicator = " 🌟" if result.priority_override else ""

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}{override_indicator}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception:
        return False


print("✅ Condition functions defined")

## Trin 6: Opret Tilpasset Display Executor

Samme executor som før - viser den endelige workflow-udgang.


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """Display the final result as workflow output."""
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created")

## Trin 7: Indlæs miljøvariabler

Konfigurer LLM-klienten (GitHub Models eller OpenAI).


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Trin 8: Opret AI-agenter med Middleware

**VIGTIG FORSKEL:** Når vi opretter availability_agent, giver vi parameteren `middleware`!

Dette er, hvordan vi injicerer priority_check_middleware i agentens funktionskaldspipeline.


In [ ]:
# Agent 1: Check availability with tool + middleware
availability_agent = AgentExecutor(
    await provider.create_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), message (string), "
            "and priority_override (bool, true if priority member got special access). "
            "The message should summarize the availability status and mention if priority override occurred."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
        middleware=[priority_check_middleware],  # 🌟 MIDDLEWARE INJECTION!
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    await provider.create_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    await provider.create_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "If priority_override is true in the input, mention that they received priority member access. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - WITH priority_check_middleware 🌟</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## Trin 9: Byg arbejdsgangen

Samme arbejdsgangsstruktur som før - betinget routing baseret på tilgængelighed.


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH (can be triggered by middleware override!)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing (Middleware-Aware):</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> (or 🌟 <strong>priority override</strong>) → booking_agent → display_result
        </p>
    </div>
""")
)

## Step 10: Test Case 1 - Almindelig bruger i Paris (Ingen overskrivelse)

En almindelig bruger forsøger at booke Paris → Ingen ledige værelser → Rutes til alternative_agent


In [ ]:
# Set as regular user
set_user("regular_user")

display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Regular User + Paris</h3>
        <p style='margin: 0;'><strong>Expected:</strong> No rooms → No middleware override → Alternative suggestion</p>
    </div>
""")
)

# Create request
request_regular = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_regular = await workflow.run(request_regular)
outputs_regular = events_regular.get_outputs()

# Display results
if outputs_regular:
    result_regular = AlternativeResult.model_validate_json(outputs_regular[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: #fff; border: 2px solid #ff9800; border-radius: 12px; margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #e65100;'>📊 RESULT (Regular User)</h3>
            <div style='background: #fff3e0; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0;'><strong>Middleware:</strong> No priority override (regular user)</p>
                <p style='margin: 0 0 10px 0;'><strong>Alternative:</strong> 🏨 {result_regular.alternative_destination}</p>
                <p style='margin: 0;'><strong>Reason:</strong> {result_regular.reason}</p>
            </div>
        </div>
    """)
    )

## Step 11: Test Case 2 - 🌟 Prioritetsbruger i Paris (MED Override!)

Et prioritetsmedlem prøver at booke Paris → Ingen værelser i starten → 🌟 Middleware tilsidesætter! → Ruter til booking_agent

**Dette er den centrale demonstration af middleware-kraft!**


In [ ]:
# Set as priority user
set_user("priority_user")

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #333;'>🧪 TEST CASE 2: 🌟 Priority User + Paris</h3>
        <p style='margin: 0; color: #555;'><strong>Expected:</strong> No rooms → 🌟 MIDDLEWARE OVERRIDE → Rooms available → Booking suggestion!</p>
    </div>
""")
)

# Create request
request_priority = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_priority = await workflow.run(request_priority)
outputs_priority = events_priority.get_outputs()

# Display results
if outputs_priority:
    result_priority = BookingConfirmation.model_validate_json(outputs_priority[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px;
                    box-shadow: 0 8px 16px rgba(255,165,0,0.4); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 RESULT (Priority Member) 🌟</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Priority Override!)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> 🌟 OVERRIDE ACTIVATED!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_priority.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_priority.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_priority.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #fff3cd; border-radius: 6px; border-left: 4px solid #FF6B35;'>
                    <strong>💡 What Just Happened:</strong><br>
                    1. hotel_booking tool returned "no availability"<br>
                    2. priority_check_middleware intercepted the result<br>
                    3. Middleware checked user status: priority_user ✅<br>
                    4. Middleware OVERRODE the result to "available"<br>
                    5. Workflow routed to booking_agent instead of alternative_agent!
                </div>
            </div>
        </div>
    """)
    )

## Trin 12: Test Case 3 - Prioritetsbruger i Stockholm (Allerede tilgængelig)

Prioritetsbruger prøver Stockholm → Værelser tilgængelige → Ingen overstyring nødvendig → Ruter til booking_agent

Dette viser, at middleware kun handler, når det er nødvendigt!


In [ ]:
# Priority user is still set from previous test

display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 3: Priority User + Stockholm</h3>
        <p style='margin: 0;'><strong>Expected:</strong> Rooms available → No override needed → Booking suggestion</p>
    </div>
""")
)

# Create request
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px;
                    box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 RESULT (Priority User - No Override Needed)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Natural)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> No override needed</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #e8f5e9; border-radius: 6px; border-left: 4px solid #4caf50;'>
                    <strong>💡 Middleware Behavior:</strong><br>
                    • hotel_booking returned "available" naturally<br>
                    • Middleware saw available = true → No override needed<br>
                    • Workflow proceeded normally to booking_agent
                </div>
            </div>
        </div>
    """)
    )

## Vigtige pointer og middleware-koncept

### ✅ Det, du har lært:

#### **1. Funktionsbaseret middleware-mønster**

Middleware opsnapper funktionskald ved hjælp af en simpel async funktion:

```python
async def my_middleware(
    context: FunctionInvocationContext,
    next: Callable,
) -> None:
    # Før funktionens udførelse
    print("Intercepting...")
    
    # Udfør funktionen
    await next(context)
    
    # Efter funktionens udførelse - undersøg resultatet
    if context.result:
        # Ændr resultatet om nødvendigt
        context.result = modified_value
```

#### **2. Adgang til context og overskrivning af resultat**

- `context.function` - Få adgang til den kaldte funktion
- `context.arguments` - Læs funktionsargumenter
- `context.kwargs` - Få adgang til ekstra parametre
- `await next(context)` - Udfør funktionen
- `context.result` - Læs/ændr funktionens output

#### **3. Implementering af forretningslogik**

Vores middleware implementerer fordelene for prioritetsmedlemmer:
- **Almindelige brugere**: Ingen ændringer, standard arbejdsgang
- **Prioritetsbrugere**: Overskriv "ingen tilgængelighed" → "tilgængelig"
- **Betinget logik**: Overskriver kun når nødvendigt

#### **4. Samme arbejdsgang, forskellige resultater**

Middleware styrke:
- ✅ Ingen ændringer i arbejdsgangsstrukturen
- ✅ Ingen ændringer i værktøjsfunktionen
- ✅ Ingen ændringer i betinget routing-logik
- ✅ Kun middleware → Forskellig adfærd!

### 🚀 Anvendelser i virkeligheden:

1. **VIP/Premium-funktioner**
   - Overskriv grænser for premiumbrugere
   - Giv prioriteret adgang til ressourcer
   - Lås premium-funktioner op dynamisk

2. **A/B-test**
   - Rute brugere til forskellige implementeringer
   - Test nye funktioner med specifikke brugere
   - Gradvis udrulning af funktioner

3. **Sikkerhed og overholdelse**
   - Revisionsspor af funktionskald
   - Bloker følsomme handlinger
   - Håndhæv forretningsregler

4. **Performanceoptimering**
   - Cache resultater for specifikke brugere
   - Spring dyre operationer over når muligt
   - Dynamisk ressourceallokering

5. **Fejlhåndtering og genforsøg**
   - Fang og håndter fejl på en elegant måde
   - Implementer logik til genforsøg
   - Fald tilbage til alternative implementeringer

6. **Logging og overvågning**
   - Spor funktionsudførelsestid
   - Log parametre og resultater
   - Overvåg brugsmønstre

### 🔑 Centrale forskelle fra dekoratører:

| Funktion | Dekoratør | Middleware |
|---------|-----------|------------|
| **Omfang** | Enkel funktion | Alle funktioner i agent |
| **Fleksibilitet** | Fast ved definition | Dynamisk ved kørsel |
| **Context** | Begrænset | Fuld agent context |
| **Sammensætning** | Flere dekoratører | Middleware pipeline |
| **Agent-bevidst** | Nej | Ja (adgang til agent-tilstand) |

### 📚 Hvornår man bruger middleware:

✅ **Brug middleware når:**
- Du skal ændre adfærd baseret på bruger-/sessionstilstand
- Du vil anvende logik på flere funktioner
- Du har brug for adgang til agent-niveau context
- Du implementerer tværgående bekymringer (logging, auth osv.)

❌ **Brug ikke middleware når:**
- Simpel inputvalidering (brug Pydantic)
- Funktionsspecifik logik (hav det i funktionen)
- Éngangsændringer (ændr bare funktionen)

### 🎓 Avancerede mønstre:

```python
# Flere middleware (udførelsesrækkefølge betyder noget!)
middleware=[
    logging_middleware,      # Logger først
    auth_middleware,         # Tjekker derefter auth
    cache_middleware,        # Tjekker derefter cache
    rate_limit_middleware,   # Derefter ratebegrænsninger
    priority_check_middleware  # Endelig prioritetstjek
]

# Betinget middleware-udførelse
async def conditional_middleware(context, next):
    if should_execute(context):
        await next(context)
        # Ændrer resultat
    else:
        # Spring udførelse helt over
        context.result = cached_value
```

### 🔗 Relaterede koncepter:

- **Agent Middleware**: Opsnapper agent.run() kald
- **Funktions Middleware**: Opsnapper værktøjsfunktionskald (det vi brugte!)
- **Middleware Pipeline**: Kæde af middleware udført i rækkefølge
- **Context Propagation**: Videregiv tilstand gennem middleware-kæden


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokument er blevet oversat ved hjælp af AI-oversættelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selvom vi bestræber os på nøjagtighed, skal du være opmærksom på, at automatiserede oversættelser kan indeholde fejl eller unøjagtigheder. Det originale dokument på dets oprindelige sprog bør betragtes som den autoritative kilde. For kritisk information anbefales professionel menneskelig oversættelse. Vi påtager os intet ansvar for misforståelser eller fejltolkninger, der opstår som følge af brugen af denne oversættelse.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
